# Cell type phenotyping and annotation

Explore the full pancreas cohort using the updated Islet Explorer protein assignments. The same UMAP shows where cell identities, disease groups, and protein signals overlap. The heatmap and dotplots summarize the proteins that distinguish each phenotype.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/02_phenotyping.ipynb).

## 1. Load the processed PhenoCycler data

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding. The cohort UMAP sample enriches uncommon cell types. Marker summaries and cell proportions use all admitted cells.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
LEVEL = "primary"                    # primary, detail, immune_class, t_subtype, macrophage_profile
UMAP_MARKERS = ["GCG", "INS", "SST", "Podoplanin", "B3TUBB", "CD56", "CD31", "Ker8_18", "CD3e"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
atlas = sc.read_h5ad(data / "pancreas_umap.h5ad")
coords = np.asarray(atlas.obsm["X_umap"])
mapped = atlas[np.isfinite(coords).all(axis=1)].copy()
print(f"{len(manifest)} pancreas donors · {manifest.cells.sum():,} admitted cells · {mapped.n_obs:,} displayed on the joint UMAP")
print("Disease groups:", ", ".join(f"{g} ({n} donors)" for g,n in manifest.groupby("disease").size().items()))
type_order = ["Alpha","Beta","Delta","Endocrine (ambiguous subtype)","Endocrine (unresolved subtype)",
              "Exocrine","Immune","Blood vessel","Lymphatic","Nerve","Unresolved (no lineage evidence)"]
inks = {"Alpha":"#cf6c30", "Beta":"#287cac", "Delta":"#4c8b58", "Immune":"#965084",
        "Exocrine":"#c1b7a6", "Endocrine (ambiguous subtype)":"#bda559",
        "Endocrine (unresolved subtype)":"#99853f", "Blood vessel":"#b34b50",
        "Lymphatic":"#68a69a", "Nerve":"#5b558c", "Unresolved (no lineage evidence)":"#868686"}
for label in type_order:
    inks.setdefault(str(label), "#73777d")

## 2. Visualize cell types on the shared protein UMAP

All panels use the same embedding and limits. Select a disease group or donor above to inspect the same population on those coordinates.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.6))
limits = [(coords[np.isfinite(coords[:,k]), k].min(), coords[np.isfinite(coords[:,k]), k].max()) for k in (0,1)]
sc.pl.umap(mapped, color="disease", palette=group_ink, size=3, ax=axes[0,0],
           title="Disease group", legend_loc="none", show=False)
for view, ax, title in [(mapped, axes[0,1], "All donors"),
                       (mapped[mapped.obs.disease.astype(str) == DISEASE].copy(), axes[1,0], DISEASE),
                       (mapped[mapped.obs.donor.astype(str) == DONOR].copy(), axes[1,1], f"Donor {DONOR}")]:
    sc.pl.umap(view, color="primary", palette=inks, ax=ax, size=3 if view.n_obs>8000 else 8,
               title=f"{title} · {view.n_obs:,} cells", legend_loc="none", show=False)
for ax in axes.flat:
    ax.set(xlim=limits[0], ylim=limits[1], aspect="equal")
for ax, palette, labels, ncol in [(axes[0,0], group_ink, group_order, 3),
                                   (axes[1,0], inks, mapped.obs.primary.dropna().unique(), 3)]:
    handles=[Line2D([],[],marker="o",linestyle="",markersize=5,color=palette[v],label=DISPLAY_NAMES.get(v,v)) for v in labels]
    if ax is axes[0,0]:
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.50,-.01), ncol=ncol, frameon=False, fontsize=8)
    else:
        fig.legend(handles=handles, loc="lower center", ncol=ncol, frameon=False, fontsize=8)
fig.subplots_adjust(top=.95, bottom=.15, hspace=.25)
plt.show()

## 3. Marker expression on the UMAP

Hormones, structural channels and immune identity markers share the same embedding. Gray denotes unavailable measurements. Select additional measured channels with `UMAP_MARKERS`. The heatmaps below cover all identity-marker families.

In [ ]:
show_markers = [m for m in UMAP_MARKERS if m in mapped.var_names]
sc.pl.umap(mapped, color=show_markers, layer="typing_scaled", use_raw=False, ncols=3,
           size=4, cmap="viridis", vmin=0, vmax=1, na_color="#cecece", wspace=.26, hspace=.32)

## 4. Compare protein signatures across phenotypes

Each panel uses full-section measurements. Row labels show the size of the assessable population and the number of sections with at least one assigned cell. A gray tile or × marks a missing or noncomparable measurement. Dot area is the measured fraction at or above 0.10 and color is mean scaled protein. The group and cohort default gives each contributing donor equal weight. Set `WEIGHTING = "cells"` for the pooled cell estimate. The returned tables give exact measured cells, positive cells and contributing donors for every phenotype and marker.

The named endocrine rows are separated from the saturated hormone conflicts. “CD99 endocrine, no hormone above floor” denotes the distinct CD99-supported group with all three hormones measured at the typing floor.

In [ ]:
WEIGHTING = "donors"  # "cells" pools the same measured cells across donors.
cohort_markers = show_profiles(counts, evidence, level=LEVEL, weighting=WEIGHTING)
display(cohort_markers.reset_index())

In [ ]:
group_markers = show_profiles(counts, evidence, level=LEVEL, disease=DISEASE, weighting=WEIGHTING)
donor_markers = show_profiles(counts, evidence, level=LEVEL, donor=DONOR)
display(donor_markers.reset_index())

### Endocrine calls with conflicting hormones

The native image audit groups the final unresolved cells by their measured competing pair and decision reason. Area gap is measured in percentage points of the full eroded interior. The two brightness columns use uncapped marker-specific fluorescence in that same interior. The named-cell comparisons below use the original cytoplasmic measurement compartment for **both** groups.

In [ ]:
for scope, selection in [("All donors", {}), (DISEASE, {"disease": DISEASE}), (f"Donor {DONOR}", {"donor": DONOR})]:
    print(scope)
    conflicts = conflict_summary(data, counts, **selection)
    if conflicts.empty:
        print("No remaining hormone conflicts.")
    else:
        display(conflicts)

### Compare named cells and unresolved pairs in the same donors

For each pair and each contender, compare all residual cells with that pair to all cells with the corresponding named subtype in the same sections. Both arms use identical donor sets with finite measurements in both populations. An absent named class reduces the matched donor count. The two means show how pooling cells changes the equal-donor estimate. A cell counts as positive at scaled signal ≥0.10. The uncapped measure uses the same aligned cytoplasmic signal and cohort calibration before clipping. Saturated scores selected these residual cells for review.

In [ ]:
from phenotype_signatures import endocrine_evidence, matched_hormones

pair_measurements, named_support = endocrine_evidence(data, counts)
for scope, selection in [("All donors", {}), (DISEASE, {"disease": DISEASE}), (f"Donor {DONOR}", {"donor": DONOR})]:
    print(scope)
    matched = matched_hormones(pair_measurements, **selection)
    if matched.empty:
        print("No residual pair in this selection.")
    else:
        display(matched.round(3))

### How much evidence supports the named endocrine groups?

The full-section census distinguishes a uniquely highest hormone from a direct endocrine Single and a missing donor/stratum refinement bound. These conditions can overlap. “Below 0.10” refers only to the display threshold in the measured defining hormone. A cell may have low fluorescence even when the bound is available, or a strong signal when the bound is unavailable. The intersection columns identify cells with both conditions. The last column reports the weakest finite signal. Each section’s denominator and route counts are shown beneath the cohort table.

In [ ]:
columns = ["named_cells", "measured", "unique_hormone_winner", "direct_Single", "no_refinement_bound",
           "below_0.10", "below_0.10_and_no_bound", "direct_Single_below_0.10_and_no_bound"]
grouped = named_support.groupby(["named_type", "defining_hormone"], observed=True)
totals = grouped[columns].sum()
totals["weakest_measured_signal"] = grouped.weakest_measured_signal.min()
display(totals)
display(named_support.sort_values(["disease", "donor", "named_type"]))

### Do neighboring protein profiles support the final cell identity?

The production graph compares each cell with its 30 nearest neighbors in the full donor protein representation. Concordance is the fraction sharing the final label. Below 0.40 means fewer than 12 of 30 share that label. A measured zero means no same-label neighbor. An unavailable graph measurement stays missing. The producer uses the donor- and stratum-specific defining-marker bound to review calls. Appearance on the two-dimensional UMAP alone does not decide a label.

The first table uses **all admitted cells** and shows outcomes for each final lineage. The second separates evaluated cells from those with an unavailable bound or another recorded outcome in the selected donor.

In [ ]:
from phenotype_signatures import neighbor_support, summarize_neighbors

refinement_qc = neighbor_support(data, counts)
lineage_qc = summarize_neighbors(refinement_qc, ["celltype"])
display(lineage_qc.rename(index=DISPLAY_NAMES).round(3))
selected_qc = summarize_neighbors(refinement_qc.loc[refinement_qc.donor.eq(DONOR)], ["celltype", "refine_state"])
display(selected_qc.round(3))

The two maps use the same saved coordinates. The right panel shows the selected donor on the cohort axes. Color comes directly from the production graph and gray marks missing concordance. The following counts use only displayed points. The selected donor lookup reports the identities, first-pass assignment and agreement for the lowest-agreement measured cells.

In [ ]:
agreement = "refinement_concordance"
if agreement not in mapped.obs:
    raise KeyError("The display sample must carry production refinement_concordance")
fig, axes = plt.subplots(1,2,figsize=(10.7,4.4),sharex=True,sharey=True)
summaries = []
for view, ax, title in [(mapped, axes[0], "All donors"),
                       (mapped[mapped.obs.donor.astype(str).eq(DONOR)].copy(), axes[1], f"Donor {DONOR}")]:
    numeric = pd.to_numeric(view.obs[agreement],errors="raise")
    measured = numeric.notna()
    if not numeric[measured].between(0,1).all():
        raise ValueError("UMAP point agreement lies outside 0 to 1")
    if len(view):
        sc.pl.umap(view, color=agreement, ax=ax, vmin=0, vmax=1, cmap="viridis",
                   na_color="#c2c5c6", size=6 if view is mapped else 14, show=False,
                   title=f"{title} · {measured.sum():,} graph-measured")
    ax.set(xlim=limits[0],ylim=limits[1],aspect="equal")
    summaries.append({"display sample":title,"coordinates":len(view),"graph measured":int(measured.sum()),
                      "low agreement (<0.40)":int(numeric.lt(.40).sum()),"zero agreement":int(numeric.eq(0).sum()),
                      "unmeasured":int(numeric.isna().sum())})
fig.suptitle("Fraction of protein neighbors with the same final label",fontsize=12)
fig.tight_layout();plt.show()
display(pd.DataFrame(summaries).set_index("display sample"))
lookup = mapped.obs.loc[mapped.obs.donor.astype(str).eq(DONOR),
                        ["celltype","first_pass","refine_state",agreement]].dropna(subset=[agreement])
lookup.index.name = "native_cell_id"
display(lookup.sort_values(agreement,kind="stable").head(20))

## 5. Explore marker co-expression

Each donor contributes 4,000 randomly selected cells. The panels show the measured cells in the disease group and selected donor. Axes use aligned protein intensity after log(1 + intensity). The scale retains differences above the ceiling used for typing.

In [ ]:
scatter_cells = sc.read_h5ad(data / "pancreas_random.h5ad")
pair = scatter_cells[:, list(PAIR)].copy()
raw = np.asarray(pair.X)
good = np.isfinite(raw).all(axis=1) & (raw >= 0).all(axis=1)
pair = pair[good].copy()
pair.X = np.log1p(np.asarray(pair.X))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), sharex=True, sharey=True)
for view, ax, title in [(pair[pair.obs.disease.astype(str)==DISEASE].copy(), axes[0], DISEASE),
                       (pair[pair.obs.donor.astype(str)==DONOR].copy(), axes[1], f"Donor {DONOR}")]:
    if view.n_obs:
        sc.pl.scatter(view, x=PAIR[0], y=PAIR[1], color="primary", palette=[inks[c] for c in view.obs.primary.cat.categories], size=10 if view.n_obs < 8000 else 4,
                      alpha=.72, ax=ax, show=False, use_raw=False, legend_loc="none", frameon=True)
    ax.grid(False)
    ax.set(title=f"{title} · {view.n_obs:,} measured cells", xlabel=f"log1p({PAIR[0]})", ylabel=f"log1p({PAIR[1]})")
used = set(pair.obs.loc[pair.obs.disease.astype(str).eq(DISEASE) | pair.obs.donor.astype(str).eq(DONOR),"primary"].dropna())
handles=[Line2D([],[],marker="o",linestyle="",markersize=5,color=inks[label],label=DISPLAY_NAMES.get(label,label))
         for label in type_order if label in used]
fig.legend(handles=handles,loc="lower center",ncol=4,frameon=False,fontsize=8)
fig.subplots_adjust(bottom=.29, wspace=.22)
plt.show()

Full donor files contain all 68 marker columns with availability and compartment metadata, both expression scales, original and updated cell types, and native coordinates. Change the field in the optional example to use other Scanpy functions.

In [ ]:
donor = sc.read_h5ad(data / "pancreas" / f"{DONOR}.h5ad", backed="r")
print(donor)
display(donor.obs["celltype"].value_counts().rename(index=DISPLAY_NAMES).rename("admitted cells").to_frame())
# For your own Scanpy plot, load just the desired cells/markers from the backed object:
# small = donor[:10000, ["GCG","INS","SST"]].to_memory()
# sc.pl.violin(small, ["GCG","INS","SST"], groupby="celltype", layer="typing_scaled")

Continue with [spatial analysis and donor trajectories](03_spatial_analysis.ipynb) or [group comparisons and proportions](04_group_comparisons.ipynb).